# CCT College Dublin — Continuous Assessment 1 (CA1)
## Machine Learning (Sept 2026 - FT)
**Module:** Machine Learning  
**Lecturer:** Dr. Muhammad Iqbal  
**Programme:** HDip in Science in Data Analytics for Business  
**Domain:** Healthcare (`processed.cleveland.data` / `heart.csv` — Official UCI Cleveland Heart Disease Dataset, $N=303$, $d=14$)  

---
### Why I Chose This Problem & How Each of the 5 Classifiers Fits Our Clinical Data
When I started planning this CA1 assignment, I wanted a domain where classification metrics actually mean something tangible. In clinical cardiology triage, a **false negative** (telling a patient with coronary narrowing that they are fine) can delay life-saving intervention, whereas a **false positive** simply leads to a follow-up stress test.

Because our dataset mixes continuous physiological variables (`age`, `trestbps`, `chol`, `thalach`, `oldpeak`) with encoded clinical categories (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`), no single algorithm is guaranteed to dominate a priori (Wolpert's No Free Lunch theorem). Therefore, I selected five classifiers that test distinct mathematical assumptions on this clinical feature space:
1. **Logistic Regression**: Serves as our parametric linear baseline. It models the log-odds of heart disease as an additive linear combination of predictors.
2. **k-Nearest Neighbors ($k$-NN, $k=5$)**: A non-parametric, instance-based learner. It assumes patients with similar standardised physiological profiles share the same diagnosis, capturing local cluster structure.
3. **Decision Tree (CART)**: Builds hierarchical, axis-aligned clinical thresholds (e.g., splitting on `thal` and `cp`). While intuitive for clinicians, unconstrained trees tend to memorise noise (high variance).
4. **Random Forest**: An ensemble of decorrelated trees trained via bootstrap aggregation (*bagging*) and random feature subspaces. It fits mixed clinical data exceptionally well by capturing non-linear interactions while suppressing single-tree variance.
5. **Support Vector Machine (SVC)**: Constructs a maximum-margin separating hyperplane (using linear or RBF kernel projections) to separate overlapping high-risk and low-risk patient profiles in 13-dimensional space.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

RANDOM_STATE = 42
sns.set_style('whitegrid')
print('Libraries imported successfully.')

Libraries imported successfully.


## 1. Raw Data Ingestion & In-Notebook Pre-processing (`processed.cleveland.data` $\rightarrow$ `heart.csv`)
When I inspected the raw `processed.cleveland.data` file downloaded from the UCI Machine Learning Repository, I noticed three practical data-cleaning challenges:
1. **No Header Row**: The raw file starts directly with patient numbers (`63.0, 1.0, 1.0, 145.0...`), so we must assign the 14 official clinical attribute names documented by Detrano et al. (1989).
2. **Missing Values Encoded as `'?'`**: Specifically, 4 records in `ca` (major vessels colored by fluoroscopy) and 2 records in `thal` (thalassemia defect) contain `'?'`. If we blindly run `.dropna()`, our dataset shrinks from **303 rows to 297 rows**—which would violate the CA1 requirement of $>300$ instances! Because both `ca` and `thal` are discrete clinical categories, I imputed the 6 missing entries using their respective **clinical modes** (`ca = 0.0`, `thal = 3.0`), preserving all 303 patients.
3. **Multi-class Target (`num` 0–4)**: In the raw UCI file, `0` represents healthy controls ($<50\%$ coronary narrowing), while `1, 2, 3, 4` represent degrees of coronary artery disease ($>50\%$ narrowing). Following standard clinical benchmark practice, I binarised `target = (num > 0).astype(int)` and exported the clean table to `heart.csv`.

In [2]:
uci_columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'
]

try:
    raw_df = pd.read_csv('processed.cleveland.data', names=uci_columns)
    print(f'Raw UCI processed.cleveland.data loaded: {raw_df.shape}')
    print("Missing '?' strings detected per column:")
    print((raw_df == '?').sum()[lambda x: x > 0])
    
    # Replace '?' with NaN and convert all columns to numeric
    df = raw_df.replace('?', np.nan).apply(pd.to_numeric)
    
    # Mode imputation for discrete clinical variables 'ca' and 'thal' to preserve all 303 rows
    for col in ['ca', 'thal']:
        n_miss = df[col].isnull().sum()
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"\nImputed '{col}' ({n_miss} missing) with clinical mode = {mode_val}")
        
    # Binarize target: 0 = Healthy (<50% narrowing), 1 = Heart Disease (>50% narrowing)
    df['target'] = (df['num'] > 0).astype(int)
    df = df.drop(columns=['num'])
    
    # Export clean dataset to heart.csv for full reproducibility
    df.to_csv('heart.csv', index=False)
    print(f"Clean dataset saved to 'heart.csv' -> Final shape: {df.shape} | Total NaNs: {df.isnull().sum().sum()}")
except FileNotFoundError:
    df = pd.read_csv('heart.csv')
    print(f'Pre-cleaned heart.csv loaded directly: {df.shape}')

y = df['target']
print(f'Target Class Balance: 0 (Healthy) = {(y==0).sum()} ({(y==0).mean()*100:.2f}%) | 1 (Heart Disease) = {(y==1).sum()} ({(y==1).mean()*100:.2f}%)')
display(df.head())
display(df.describe().T.round(2))

Raw UCI processed.cleveland.data loaded: (303, 14)
Missing '?' strings detected per column:
ca      4
thal    2
dtype: int64

Imputed 'ca' (4 missing) with clinical mode = 0.0
Imputed 'thal' (2 missing) with clinical mode = 3.0
Clean dataset saved to 'heart.csv' -> Final shape: (303, 14) | Total NaNs: 0
Target Class Balance: 0 (Healthy) = 164 (54.13%) | 1 (Heart Disease) = 139 (45.87%)


## 2. Feature Scaling (`StandardScaler`)
Looking at the summary statistics table above, serum cholesterol (`chol`) averages $246.69\text{ mg/dl}$ (reaching up to $564\text{ mg/dl}$), whereas exercise-induced ST depression (`oldpeak`) spans $0.0$ to $6.2$. Without standardization, high-magnitude features dominate Euclidean distance calculations in $k$-NN and kernel dot products in SVM. Here we standardize all 13 predictors to $\mu = 0, \sigma = 1$.

In [3]:
X = df.drop('target', axis=1)
y = df['target']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
print('Standardized predictor matrix X_scaled shape:', X_scaled.shape)

Standardized predictor matrix X_scaled shape: (303, 13)


## 3. Train vs. Test Accuracy, Cross-Validation & Overfitting Diagnosis Across Splits (5%, 15%, 25%)
To properly diagnose **overfitting** (high variance) versus **underfitting** (high bias), we compare **Training Accuracy**, **5-Fold Cross-Validation Accuracy**, and **Hold-out Test Accuracy** across the three required splits (95/5, 85/15, and 75/25).

In [4]:
test_ratios = [0.05, 0.15, 0.25]
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE),
    'SVM (RBF)': SVC(probability=True, random_state=RANDOM_STATE)
}

split_results = []
for test_size in test_ratios:
    train_pct = int(round((1 - test_size) * 100))
    test_pct = int(round(test_size * 100))
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )
    for name, model in models.items():
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
        model.fit(X_train, y_train)
        train_acc = accuracy_score(y_train, model.predict(X_train))
        test_acc = accuracy_score(y_test, model.predict(X_test))
        split_results.append({
            'Model': name,
            'Split (Train/Test)': f'{train_pct}/{test_pct}',
            'Test Ratio (%)': test_pct,
            'Train Accuracy': round(train_acc, 4),
            'CV Mean Accuracy': round(cv_scores.mean(), 4),
            'Test Accuracy': round(test_acc, 4),
            'Generalization Gap': round(train_acc - test_acc, 4)
        })

results_df = pd.DataFrame(split_results)
print('=== Authentic UCI Cleveland: Train vs. Test Accuracy Across Splits (5%, 15%, 25%) ===')
print(results_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sub_15 = results_df[results_df['Test Ratio (%)'] == 15].melt(
    id_vars='Model', value_vars=['Train Accuracy', 'Test Accuracy'], var_name='Metric', value_name='Accuracy'
)
sns.barplot(data=sub_15, x='Model', y='Accuracy', hue='Metric', palette=['#1F4E79', '#5B9BD5'], ax=axes[0])
axes[0].set_title('Train vs. Test Accuracy (85/15 Split — Overfitting Check)', fontsize=11, fontweight='bold')
axes[0].set_ylim(0.5, 1.08)
axes[0].tick_params(axis='x', rotation=20)

sns.lineplot(data=results_df, x='Test Ratio (%)', y='Test Accuracy', hue='Model', marker='o', linewidth=2.2, ax=axes[1])
axes[1].set_title('Hold-out Test Accuracy Across Splits (5%, 15%, 25%)', fontsize=11, fontweight='bold')
axes[1].set_xticks([5, 15, 25])
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

=== Authentic UCI Cleveland: Train vs. Test Accuracy Across Splits (5%, 15%, 25%) ===
                  Model Split (Train/Test)  Test Ratio (%)  Train Accuracy  CV Mean Accuracy  Test Accuracy  Generalization Gap
0   Logistic Regression               95/5               5          0.8502            0.8292         0.7500              0.1002
1            k-NN (k=5)               95/5               5          0.8850            0.8256         0.8125              0.0725
2         Decision Tree               95/5               5          1.0000            0.7385         0.6250              0.3750
3         Random Forest               95/5               5          1.0000            0.8188         0.8125              0.1875
4             SVM (RBF)               95/5               5          0.8990            0.8256         0.7500              0.1490
5   Logistic Regression              85/15              15          0.8444            0.8287         0.8478             -0.0035
6            k-NN 

### Critical Discussion on Overfitting, Underfitting, and Generalization
* **Severe Overfitting in Unpruned Decision Tree**: The single Decision Tree achieves `1.0000` (100%) Training Accuracy across all splits, yet drops to `0.6250` (5% test), `0.7391` (15% test), and `0.7237` (25% test)—a **26.1% to 37.5% generalization gap**. Without depth limits, the tree grows until every patient sits in a pure leaf, memorizing noise.
* **Absence of Underfitting in Linear Baseline**: Logistic Regression achieves `0.8444` Train Accuracy and `0.8478` Test Accuracy at the 85/15 split (`-0.0035` gap), demonstrating low variance and strong linear generalization.
* **Sample-Size Volatility at 5% Test Split**: At 95/5, the test fold contains only $16$ patients, so 3 borderline cases shift accuracy by nearly $19\%$. Expanding the test fold to 15% ($46$ patients) and 25% ($76$ patients) stabilizes out-of-sample evaluation.

## 4. Hyperparameter Tuning via `GridSearchCV` (80/20 Stratified Split)
To eliminate the `1.0000` training memorization of unpruned trees in Random Forest and optimize the margin hyperplane in SVM, we execute 5-fold `GridSearchCV` on a stratified 80/20 split.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

rf_param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 8, None], 'min_samples_split': [2, 5, 10]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), rf_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_

print('--- Tuned Random Forest (80/20 Split) ---')
print('Optimal Params:', rf_grid.best_params_)
print(f'Train Accuracy: {best_rf.score(X_train, y_train):.4f} | 5-Fold CV Accuracy: {rf_grid.best_score_:.4f} | Hold-Out Test Accuracy: {best_rf.score(X_test, y_test):.4f}\n')

svm_param_grid = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 'auto']}
svm_grid = GridSearchCV(SVC(probability=True, random_state=RANDOM_STATE), svm_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_train, y_train)
best_svm = svm_grid.best_estimator_

print('--- Tuned Support Vector Machine (SVC) ---')
print('Optimal Params:', svm_grid.best_params_)
print(f'Train Accuracy: {best_svm.score(X_train, y_train):.4f} | 5-Fold CV Accuracy: {svm_grid.best_score_:.4f} | Hold-Out Test Accuracy: {best_svm.score(X_test, y_test):.4f}')

--- Tuned Random Forest (80/20 Split) ---
Optimal Params: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}
Train Accuracy: 0.9215 | 5-Fold CV Accuracy: 0.8384 | Hold-Out Test Accuracy: 0.9016

--- Tuned Support Vector Machine (SVC) ---
Optimal Params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Train Accuracy: 0.8554 | 5-Fold CV Accuracy: 0.8304 | Hold-Out Test Accuracy: 0.8852


## 5. Confusion Matrices & Statistical Reasoning on Feature Importance
Because Random Forest **Gini Importance** (Mean Decrease in Impurity) can slightly favor continuous features with many split points (`age`, `chol`, `trestbps`), I cross-checked Gini rankings against non-parametric **Spearman rank correlation ($r_s$)** and two-tailed **hypothesis test $p$-values**.

In [6]:
y_pred_rf = best_rf.predict(X_test)
y_pred_svm = best_svm.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0])
axes[0].set_title('Confusion Matrix: Tuned Random Forest (90.16%)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[1])
axes[1].set_title('Confusion Matrix: Tuned SVM (88.52%)', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

# Statistical Feature Importance Verification (Gini + Spearman r + p-values)
importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
corrs = X.apply(lambda col: stats.spearmanr(col, y)[0]).loc[importances.index]
pvals = X.apply(lambda col: stats.spearmanr(col, y)[1]).loc[importances.index]

stat_summary = pd.DataFrame({
    'Gini Importance': importances.round(4),
    'Spearman r': corrs.round(4),
    'p-value': pvals.apply(lambda p: f'{p:.2e}' if p < 0.001 else f'{p:.4f}')
})
print('=== Authentic UCI Cleveland: Gini Importance vs. Spearman Rank Correlation ===')
print(stat_summary.head(10).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
importances.head(8).plot(kind='barh', color='#1F4E79', ax=axes[0])
axes[0].set_title('Random Forest Gini Importance (Top 8)', fontweight='bold')
axes[0].invert_yaxis()

corrs.head(8).abs().plot(kind='barh', color='#2E75B6', ax=axes[1])
axes[1].set_title('Absolute Spearman Correlation |r| with Target', fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

=== Authentic UCI Cleveland: Gini Importance vs. Spearman Rank Correlation ===
          Gini Importance  Spearman r   p-value
thal               0.1789      0.5187  2.87e-22
cp                 0.1422      0.4720  3.23e-18
thalach            0.1238     -0.4235  1.29e-14
ca                 0.1170      0.4881  1.52e-19
oldpeak            0.0910      0.4134  6.18e-14
exang              0.0625      0.4319  3.37e-15
trestbps           0.0590      0.1282    0.0257
age                0.0577      0.2367  3.15e-05
chol               0.0561      0.1211    0.0351
slope              0.0490      0.3644  6.00e-11
